# Built-In PTQ Baseline: ResNet-18

This notebook applies TensorFlow/TFLite post-training quantization to the pretrained ResNet-18 model and measures size, weight memory, activation memory, tensor types, and sample prediction output.


## Setup

In [ ]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT

In [ ]:
import importlib

import numpy as np
import pandas as pd
from PIL import Image
from tensorflow import keras

from src.models import load_pretrained_resnet18
from src.quantization import (
    build_fixed_input_model,
    convert_dynamic_range,
    convert_float,
    convert_full_integer,
)
import src.evaluation.tflite_metrics as tflite_metrics

importlib.reload(tflite_metrics)

inspect_tflite = tflite_metrics.inspect_tflite
predict_tflite = tflite_metrics.predict_tflite
reduction_metrics = tflite_metrics.reduction_metrics
size_metrics = tflite_metrics.size_metrics


## Load CIFAR-10 Dataset

CIFAR-10 images are used as representative inputs for full INT8 calibration and metric prediction. The pretrained ResNet-18 model is still ImageNet-trained, so CIFAR-10 labels are not used for accuracy; they are only retained as dataset metadata.


In [ ]:
(cifar_train_images, cifar_train_labels), _ = keras.datasets.cifar10.load_data()

num_cifar_samples = 100
raw_images = [
    np.asarray(
        Image.fromarray(image).resize((224, 224), Image.Resampling.BILINEAR),
        dtype=np.uint8,
    )[None, ...]
    for image in cifar_train_images[:num_cifar_samples]
]
cifar_labels = cifar_train_labels[:num_cifar_samples].reshape(-1)

len(raw_images), raw_images[0].shape, raw_images[0].dtype, cifar_labels[:10]


## Load ResNet-18 And Preprocess Samples

In [ ]:
model = load_pretrained_resnet18()
fixed_model = build_fixed_input_model(model)

samples = [np.asarray(model.preprocessor(image), dtype=np.float32) for image in raw_images]
samples[0].shape, samples[0].dtype

## Convert Models

We create three TFLite models:

1. FP32 baseline
2. Dynamic-range INT8 PTQ: weights quantized, input/output stay FP32
3. Full INT8 PTQ: weights, intermediate activations, model input, and model output quantized to INT8


In [ ]:
output_dir = PROJECT_ROOT / "artifacts" / "resnet18_builtin_ptq_notebook"
output_dir.mkdir(parents=True, exist_ok=True)

paths = {
    "float32": output_dir / "resnet18_float32.tflite",
    "dynamic_int8": output_dir / "resnet18_dynamic_int8.tflite",
    "full_int8": output_dir / "resnet18_full_int8.tflite",
}

def representative_dataset():
    for sample in samples:
        yield [sample]

paths["float32"].write_bytes(convert_float(fixed_model))
paths["dynamic_int8"].write_bytes(convert_dynamic_range(fixed_model))
paths["full_int8"].write_bytes(convert_full_integer(fixed_model, representative_dataset))

{name: path.stat().st_size for name, path in paths.items()}

In [ ]:
# Check whether built-in PTQ preserves the FP32 model prediction on the first CIFAR-10 sample.
# This is prediction consistency, not CIFAR-10 accuracy: the model is pretrained on ImageNet.
fp32_output = model(samples[0], training=False).numpy()
fp32_top_class = int(np.argmax(fp32_output[0]))

ptq_predictions = []
for name in ["dynamic_int8", "full_int8"]:
    prediction = predict_tflite(paths[name], samples[0])
    ptq_top_class = prediction["top_class_index"]
    ptq_predictions.append(
        {
            "model": name,
            "dataset": "cifar10",
            "cifar10_label": int(cifar_labels[0]),
            "fp32_top_class_index": fp32_top_class,
            "ptq_top_class_index": ptq_top_class,
            "matches_fp32_prediction": ptq_top_class == fp32_top_class,
        }
    )

pd.DataFrame(ptq_predictions)


## Measure Metrics

In [ ]:
results = {}

for name, path in paths.items():
    metrics = {
        "dataset": "cifar10",
        "num_calibration_samples": len(samples),
        **inspect_tflite(path),
        **predict_tflite(path, samples[0]),
    }

    if name == "float32":
        size_bytes = path.stat().st_size
        metrics.update({
            "size_bytes": size_bytes,
            "size_mib": size_bytes / (1024 ** 2),
            "compression_ratio": 1.0,
            "memory_reduction_percent": 0.0,
        })
    else:
        metrics.update(size_metrics(paths["float32"], path))

    results[name] = metrics

baseline = results["float32"]
for metrics in results.values():
    metrics["weight_memory"] = {
        "size_bytes": metrics["weight_storage_bytes"],
        **reduction_metrics(baseline["weight_storage_bytes"], metrics["weight_storage_bytes"]),
    }
    metrics["activation_memory"] = {
        "tensor_storage_bytes": metrics["activation_tensor_storage_bytes"],
        "largest_tensor_bytes": metrics["largest_activation_tensor_bytes"],
        **reduction_metrics(
            baseline["activation_tensor_storage_bytes"],
            metrics["activation_tensor_storage_bytes"],
        ),
    }

results


## Results Table

In [ ]:
table = pd.DataFrame([
    {
        "model": name,
        "dataset": metrics["dataset"],
        "num_calibration_samples": metrics["num_calibration_samples"],
        "size_mib": metrics["size_mib"],
        "serialized_compression": metrics["compression_ratio"],
        "serialized_reduction_%": metrics["memory_reduction_percent"],
        "weight_reduction_%": metrics["weight_memory"]["memory_reduction_percent"],
        "activation_reduction_%": metrics["activation_memory"]["memory_reduction_percent"],
        "top_class_index": metrics["top_class_index"],
        "input_dtype": metrics["input_dtype"],
        "output_dtype": metrics["output_dtype"],
    }
    for name, metrics in results.items()
])

table


In [ ]:
results_path = output_dir / "results.json"
results_path.write_text(json.dumps(results, indent=2) + "\n", encoding="utf-8")
results_path

## Notes

- CIFAR-10 is used as representative input data for calibration and metric collection.
- The model is pretrained on ImageNet, so CIFAR-10 labels are not used for accuracy.
- Dynamic-range INT8 mainly reduces weight storage and keeps FP32 model input/output.
- Full INT8 reduces both weight storage and intermediate activation tensor storage, and uses INT8 model input/output.
- The activation value is a graph-level tensor-storage estimate, not exact peak runtime memory, because TFLite may reuse buffers.
